In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import MBart50Tokenizer, MBartForConditionalGeneration, Trainer, TrainingArguments

# Step 1: Creating a function toLoad dataset

In [2]:
def load_data(file_path, tokenizer, max_length=128, test_size=0.2):
    data = pd.read_csv(file_path)
    train_data, val_data = train_test_split(data, test_size=test_size)
    return Seq2SeqDataset(train_data, tokenizer, max_length), Seq2SeqDataset(val_data, tokenizer, max_length)

# Step 2: Define dataset class

In [3]:
class Seq2SeqDataset(Dataset):
    def __init__(self, data, tokenizer, max_length):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        source_text = str(self.data.iloc[idx, 0])  # Error sentence
        target_text = str(self.data.iloc[idx, 1])  # Correct sentence
        
        source = self.tokenizer(source_text, padding="max_length", truncation=True, max_length=self.max_length, return_tensors="pt")
        target = self.tokenizer(target_text, padding="max_length", truncation=True, max_length=self.max_length, return_tensors="pt")
        
        return {
            "input_ids": source["input_ids"].squeeze(),
            "attention_mask": source["attention_mask"].squeeze(),
            "labels": target["input_ids"].squeeze()
        }

# Step 3: Initialize tokenizer and model

In [4]:
tokenizer = MBart50Tokenizer.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
tokenizer.src_lang = "ta_IN"
tokenizer.tgt_lang = "ta_IN"

In [5]:
model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")

# Step 4: Load dataset

In [6]:
train_dataset, val_dataset = load_data("dataForSeq2Seq30k.csv", tokenizer)

# Step 5: Define training arguments

In [9]:
training_args = TrainingArguments(
    output_dir="./BestMbartMayangoli",
    per_device_train_batch_size=4, 
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    learning_rate=3e-5,
    save_total_limit=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=250,
    fp16=True if torch.cuda.is_available() else False, 
)

# Step 6: Initialize Trainer

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

# Step 7: Train model

In [ ]:
trainer.train()

# Step 8: Save trained model

In [12]:
model.save_pretrained("Bestmbartmodel_Mayangoli")
tokenizer.save_pretrained("BestmbartTokenizer_Mayangoli")


('BestmbartTokenizer_Mayangoli\\tokenizer_config.json',
 'BestmbartTokenizer_Mayangoli\\special_tokens_map.json',
 'BestmbartTokenizer_Mayangoli\\sentencepiece.bpe.model',
 'BestmbartTokenizer_Mayangoli\\added_tokens.json')

# Step 9: Test the model

In [13]:
from evaluate import load

### Step 9.1: Load the trained model and tokenizer

In [14]:
tokenizer = MBart50Tokenizer.from_pretrained("BestmbartTokenizer_Mayangoli")
model = MBartForConditionalGeneration.from_pretrained("Bestmbartmodel_Mayangoli")

### Step 9.2: Load dataset (Validation set)

In [15]:
data = pd.read_csv("dataForSeq2Seq30k.csv")  # Load same dataset
_, val_data = train_test_split(data, test_size=0.2)

In [16]:
val_data_sample = val_data

In [17]:
def generate_predictions(model, tokenizer, input_texts):
    model.eval()
    predictions = []
    
    for text in input_texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        with torch.no_grad():
            output_tokens = model.generate(**inputs, max_length=128)
        pred_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        predictions.append(pred_text)
    
    return predictions

In [18]:
val_error_sentences = val_data_sample.iloc[:, 0].tolist()  # Error sentences
val_correct_sentences = val_data_sample.iloc[:, 1].tolist()  # Correct sentences

In [19]:
predicted_sentences = generate_predictions(model, tokenizer, val_error_sentences)

In [20]:
bleu_metric = load("bleu")

In [21]:
# Convert to required format
predictions_for_bleu = [pred for pred in predicted_sentences]  # Remove nested lists
references_for_bleu = [[ref] for ref in val_correct_sentences]  # Ensure correct format

In [23]:
# Compute BLEU Score
bleu_score = bleu_metric.compute(predictions=predictions_for_bleu, references=references_for_bleu)

In [24]:
# Compute Exact Match Accuracy
exact_matches = sum([1 for p, t in zip(predicted_sentences, val_correct_sentences) if p.strip() == t.strip()])
accuracy = exact_matches / len(val_correct_sentences)

In [25]:
# Print results
print(f"Exact Match Accuracy: {accuracy * 100:.2f}%")
print(f"BLEU Score: {bleu_score['bleu'] * 100:.2f}%")

Exact Match Accuracy: 89.53%
BLEU Score: 92.98%


In [26]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
    

# Hyperparameter tuning
## Define hyperparameter search space

In [27]:
learning_rates = [3e-5, 1e-4]
batch_sizes = [4, 8]
num_epochs_list = [5]

In [28]:
hyperparameter_options = {
    "per_device_train_batch_size": [4, 8],
    "num_train_epochs": [5],
    "learning_rate": [3e-5, 1e-4]
}

In [30]:
import itertools
hyperparameter_combinations = list(itertools.product(*hyperparameter_options.values()))

In [31]:
best_bleu = 0
best_accuracy = 0
best_params = None

In [33]:

def generate_predictions(model, tokenizer, input_texts, device):
    model.eval()
    predictions = []
    
    for text in input_texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        
        # Move tensors to the same device as model
        inputs = {key: value.to(device) for key, value in inputs.items()}
        
        with torch.no_grad():
            output_tokens = model.generate(**inputs, max_length=128)
        
        pred_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        predictions.append(pred_text)
    
    return predictions

In [34]:
for params in hyperparameter_combinations:
    batch_size, epochs, lr = params
    
    training_args = TrainingArguments(
        output_dir="./mbart_tamil_corrector",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        learning_rate=lr,
        save_total_limit=1,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_dir="./logs",
        logging_steps=250,
        fp16=True if torch.cuda.is_available() else False,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,  
        eval_dataset=val_dataset 
    )
    
    predicted_sentences = generate_predictions(model, tokenizer, val_error_sentences, device)
    bleu_metric = load("bleu")
    
    predictions_for_bleu = [pred for pred in predicted_sentences]
    references_for_bleu = [[ref] for ref in val_correct_sentences]
    
    bleu_score = bleu_metric.compute(predictions=predictions_for_bleu, references=references_for_bleu)
    
    # Computinggg. Ex Match Accuracy
    exact_matches = sum([1 for p, t in zip(predicted_sentences, val_correct_sentences) if p.strip() == t.strip()])
    accuracy = exact_matches / len(val_correct_sentences)
    
    if bleu_score['bleu'] > best_bleu:
        best_bleu = bleu_score['bleu']
        best_accuracy = accuracy
        best_params = params
    
    print(f"Tried params: {params}, BLEU Score: {bleu_score['bleu'] * 100:.2f}, Exact Match Accuracy: {accuracy * 100:.2f}%")

print(f"Best Hyperparameters: {best_params}, Best BLEU Score: {best_bleu * 100:.2f}, Best Accuracy: {best_accuracy * 100:.2f}%")

Tried params: (4, 5, 3e-05), BLEU Score: 92.98, Exact Match Accuracy: 89.53%
Tried params: (4, 5, 0.0001), BLEU Score: 92.98, Exact Match Accuracy: 89.53%
Tried params: (8, 5, 3e-05), BLEU Score: 92.98, Exact Match Accuracy: 89.53%
Tried params: (8, 5, 0.0001), BLEU Score: 92.98, Exact Match Accuracy: 89.53%
Best Hyperparameters: (4, 5, 3e-05), Best BLEU Score: 92.98, Best Accuracy: 89.53%
